# Model Diff Comparison with APS Viewer

This notebook shows how to use `APSViewer.compare_with(...)` to see model changes.

The demo uses the sample structural model from this repo. It uploads and translates the model, reads real `externalId` values, and creates fake added, removed, and modified changes.

In a real project, replace the fake `changes` list with data from the AEC Data Model Diff API or your own comparison workflow.

**Prereqs**
- Env vars: `CLIENT_ID`, `CLIENT_SECRET`
- Dependencies: `aps_viewer_sdk`, `aps_automation_sdk`, `requests`
- A local model file to upload and translate


In [ ]:
import os
import random
import uuid
from pathlib import Path
from typing import Literal, cast

from dotenv import load_dotenv

from aps_automation_sdk import ActivityInputParameter, translate_file_in_oss
from aps_viewer_sdk import APSViewer, ChangesInput
from aps_viewer_sdk.helper import (
    get_2lo_token,
    get_all_model_properties,
    get_metadata_viewables,
)

load_dotenv()

MODEL_DERIVATIVE_REGION = os.getenv("APS_MODEL_DERIVATIVE_REGION", "US")


In [ ]:
CLIENT_ID = os.environ.get("CLIENT_ID")
CLIENT_SECRET = os.environ.get("CLIENT_SECRET")

if not CLIENT_ID or not CLIENT_SECRET:
    raise RuntimeError("CLIENT_ID and CLIENT_SECRET must be set in environment")


## Upload and translate the sample model

This follows the same upload/translation flow as the highlight example. The output is a Viewer-ready URN that can be loaded by APS Viewer.


In [ ]:
sample_path = Path("SampleStructuralModel.rvt").resolve()

print("Starting model processing workflow...")
print(f"Uploading file: {sample_path.name}")

token = get_2lo_token(CLIENT_ID, CLIENT_SECRET)

bucket_key = uuid.uuid4().hex
object_key = f"input_{uuid.uuid4()}{sample_path.suffix}"
target_urn = f"urn:adsk.objects:os.object:{bucket_key}/{object_key}"

input_param = ActivityInputParameter(
    name="inputFile",
    localName=sample_path.name,
    verb="get",
    description="Input CAD model",
    required=True,
    is_engine_input=False,
    bucketKey=bucket_key,
    objectKey=object_key,
)

input_param.upload_file_to_oss(file_path=str(sample_path), token=token)

viewer_urn = translate_file_in_oss(
    token=token,
    bucket_key=bucket_key,
    output_object_key=object_key,
    max_wait_time=300,
    poll_interval=15,
    verbose=True,
)
print(f"Viewer URN: {viewer_urn}")


## Build fake diff changes from real model elements

The diff viewer colors elements by `externalId`.

We will do four small steps:
1. Create the viewer and pick one 3D view.
2. Read model metadata and collect real external IDs.
3. Create fake added, removed, and modified changes.
4. Store the result as `ChangesInput`.


### Create the viewer and select a 3D view

This is the target model. In this demo the base model and target model are the same file, but a real comparison would use two different versions.


In [ ]:
urn_bs64 = viewer_urn

# Use the translated model as the target model.
viewer = APSViewer(
    urn=target_urn,
    token=token,
    views_selector=True,
    region=MODEL_DERIVATIVE_REGION,
)

# Pick one 3D view. The base viewer will use this same view.
viewables = viewer.get_viewables(urn_bs64)
if not viewables:
    raise RuntimeError("No viewables returned for the translated model")

first_view = next((v for v in viewables if v.get("role") == "3d"), None)
if not first_view:
    raise RuntimeError("No 3D viewables returned for the translated model")

viewer.set_view_guid(first_view["guid"], first_view["name"], first_view["role"])
print(f"Selected view: {first_view['name']} (GUID: {first_view['guid']})")


### Read external IDs from the model

The viewer needs `externalId` values so it can find and color the changed elements.


In [ ]:
# Get the metadata view that contains element properties.
metadata_views = get_metadata_viewables(
    token, urn_bs64, region=MODEL_DERIVATIVE_REGION
)
if not metadata_views:
    raise RuntimeError("No metadata viewables returned for the translated model")

model_guid = next(
    (v["guid"] for v in metadata_views if v.get("role") == "3d"),
    metadata_views[0].get("guid"),
)
if not model_guid:
    raise RuntimeError("No valid model GUID available for metadata properties")

# Read model properties and collect unique external IDs.
payload: dict[str, object] = get_all_model_properties(
    token, urn_bs64, model_guid, region=MODEL_DERIVATIVE_REGION
)
data_raw = payload.get("data")
data: dict[str, object] = cast(
    dict[str, object], data_raw if isinstance(data_raw, dict) else payload
)
collection = data.get("collection", [])

seen: set[str] = set()
external_ids: list[str] = []
for item in collection:
    ext = item.get("externalId")
    if isinstance(ext, str) and ext and ext not in seen:
        seen.add(ext)
        external_ids.append(ext)
    if len(external_ids) == 500:
        break

print(f"Collected {len(external_ids)} external IDs")


### Create fake changes

This is only demo data. We sample 50 real elements, then assign each one a fake change type.


In [ ]:
required_change_count = 50
if len(external_ids) < required_change_count:
    raise RuntimeError(
        f"Need at least {required_change_count} external IDs for this demo"
    )

rng = random.Random(23)
sampled_external_ids = rng.sample(external_ids, required_change_count)

# Use a stable mix so the example always shows all three colors.
change_types: list[Literal["added", "removed", "modified"]] = (
    ["added"] * 17 + ["removed"] * 17 + ["modified"] * 16
)
rng.shuffle(change_types)

change_items: list[dict[str, object]] = []
for index, (external_id, change_type) in enumerate(
    zip(sampled_external_ids, change_types, strict=True)
):
    change_items.append(
        {
            "id": f"fake-{change_type}-{index}",
            "externalElementId": external_id,
            "changeType": change_type,
            "displayName": f"Fake {change_type} element {index + 1}",
            "modificationTypes": (
                ["geometry", "properties"]
                if change_type == "modified"
                else []
            ),
        }
    )

print(f"Created {len(change_items)} fake changes")


### Prepare the diff input

`ChangesInput` is the small contract consumed by `compare_with(...)`.

Here we also hide unchanged elements so the colored changes are easier to inspect.


In [ ]:
changes: ChangesInput = change_items

default_visibility: dict[
    Literal["unchanged"], Literal["visible", "hidden"]
] = {"unchanged": "hidden"}


## Show the model diff viewer

This opens the split viewer. The base side and target side start from the same 3D view.

For a real comparison, use the previous model version as `base_urn` and the current model version as `APSViewer(urn=...)`.


In [ ]:
viewer.compare_with(
    base_urn=target_urn,
    changes=changes,
    base_label="same model base",
    target_label="same model target",
    base_view_guid=first_view["guid"],
    default_mode="split",
    default_visibility=default_visibility,
    legend=True,
    list_mode="none",
    sync_views=True,
)

viewer.show()


## Result

Here is the split model diff viewer:

![Model Diff Result](../../assets/example4.png)

The left pane shows removed and modified elements from the base model. The right pane shows added and modified elements from the target model. The badges show the change counts.
